In [11]:
#Q1. REGEX
"""
Q1. U.S. ZIP Codes
Match:
- 12345
- 12345-6789
- 12345 6789
Allow hyphen or space for the +4 part.
Use word boundaries so we don't match inside longer strings.
"""
zip_regex = r"\b\d{5}(?:[- ]\d{4})?\b"


"""
Q2. Words that do NOT start with a capital letter
Words may contain internal apostrophes or hyphens:
Examples: don't, state-of-the-art
Negative lookahead ensures the first character is NOT A–Z.
"""
negation_regex = r"\b(?![A-Z])[A-Za-z]+(?:['-][A-Za-z]+)*\b"


"""
Q3. Numbers with:
- optional sign (+/-)
- optional thousands separators (commas)
- optional decimal part
- optional scientific notation (e.g., 1.23e-4)
This regex handles all combinations cleanly.
"""
number_regex = r"\b[+-]?(?:\d{1,3}(?:,\d{3})*|\d+)(?:\.\d+)?(?:[eE][+-]?\d+)?\b"


"""
Q4. Spelling variants of 'email'
Match:
- email
- e-mail
- e mail
- e–mail (en dash)
Case-insensitive.
"""
email_regex = r"(?i)\be[-\s–]?mail\b"


"""
Q5. Interjection: go, goo, gooo...
One or more 'o' allowed.
Optional punctuation at the end: ! . , ?
Match as a whole word.
"""
go_regex = r"\bgo+[!.,?]?\b"


"""
Q6. Lines ending with a question mark
Allow trailing closing quotes/brackets/spaces:
Examples: ?")   ?'   ?]   ?"   ?   ?" ]
"""
line_end_regex = r"^.*\?[)\"'\]\s]*$"


# Print all regexes to verify they loaded
print(f"zip_regex: '{zip_regex}'")
print(f"negation_regex: '{negation_regex}'")
print(f"number_regex: '{number_regex}'")
print(f"email_regex: '{email_regex}'")
print(f"go_regex: '{go_regex}'")
print(f"line_end_regex: '{line_end_regex}'")

zip_regex: '\b\d{5}(?:[- ]\d{4})?\b'
negation_regex: '\b(?![A-Z])[A-Za-z]+(?:['-][A-Za-z]+)*\b'
number_regex: '\b[+-]?(?:\d{1,3}(?:,\d{3})*|\d+)(?:\.\d+)?(?:[eE][+-]?\d+)?\b'
email_regex: '(?i)\be[-\s–]?mail\b'
go_regex: '\bgo+[!.,?]?\b'
line_end_regex: '^.*\?[)\"'\]\s]*$'


In [14]:
#Q2.Manual BPE on a toy corpus
"""
Q2. Manual BPE on a toy corpus + Mini BPE learner

Toy corpus (from class):
low low low low low lowest lowest newer newer newer newer newer newer wider wider wider new new

This cell does:
1) Shows initial vocabulary with end-of-word marker _.
2) Implements a mini BPE learner on the toy corpus:
   - Prints top pair at each step
   - Prints evolving vocabulary size
3) Segments: new, newer, lowest, widest, newestest
4) Trains BPE on a small English paragraph (for 2.3-style exploration):
   - Learns ~30 merges
   - Shows top 5 merges and 5 longest subword tokens
   - Segments 5 words from the paragraph
"""

# 2.1 – Initial corpus + vocabulary with end-of-word marker _

toy_words = "low low low low low lowest lowest newer newer newer newer newer newer wider wider wider new new".split()

# Add end-of-word marker _
toy_corpus = [" ".join(list(word) + ["_"]) for word in toy_words]

print("Toy corpus with end-of-word marker (_):")
for line in toy_corpus[:5]:
    print("  ", line)

# Initial vocabulary: characters + _
vocab = set()
for line in toy_corpus:
    for ch in line.split():
        vocab.add(ch)

print("\nInitial vocabulary (characters + _):")
print(sorted(vocab))

# Helper functions for BPE

from collections import Counter, defaultdict

def get_stats(corpus):
    """
    Compute bigram (pair) counts over the corpus.
    Each word is a sequence of tokens (split by space).
    """
    pairs = Counter()
    for line in corpus:
        symbols = line.split()
        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i+1])
            pairs[pair] += 1
    return pairs

def merge_pair(corpus, pair):
    """
    Merge a given pair (a, b) into a single token 'a+b' across the corpus.
    """
    a, b = pair
    new_corpus = []
    for line in corpus:
        symbols = line.split()
        i = 0
        merged = []
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i+1] == b:
                merged.append(a + b)
                i += 2
            else:
                merged.append(symbols[i])
                i += 1
        new_corpus.append(" ".join(merged))
    return new_corpus

def update_vocab_from_corpus(corpus):
    """
    Build vocabulary from all tokens in the corpus.
    """
    v = set()
    for line in corpus:
        for tok in line.split():
            v.add(tok)
    return v


# 2.2 – Mini BPE learner on toy corpus

def learn_bpe(corpus, num_merges=10):
    """
    Learn BPE merges on the given corpus.
    Prints:
    - top pair at each step
    - evolving vocabulary size
    Returns:
    - final corpus
    - list of merges
    """
    merges = []
    current_corpus = corpus[:]
    for step in range(1, num_merges + 1):
        stats = get_stats(current_corpus)
        if not stats:
            break
        best_pair, best_count = stats.most_common(1)[0]
        merges.append(best_pair)
        current_corpus = merge_pair(current_corpus, best_pair)
        vocab_now = update_vocab_from_corpus(current_corpus)
        print(f"Step {step}: top pair = {best_pair} (count={best_count}), vocab size = {len(vocab_now)}")
    return current_corpus, merges

print("\n=== Mini BPE on toy corpus (showing first 10 merges) ===")
final_toy_corpus, toy_merges = learn_bpe(toy_corpus, num_merges=10)

print("\nFinal toy corpus snippet after merges:")
for line in final_toy_corpus[:5]:
    print("  ", line)

print("\nLearned merges (first 10):")
for i, m in enumerate(toy_merges, 1):
    print(f"  {i}: {m}")

# Updated vocabulary after merges
toy_vocab_final = update_vocab_from_corpus(final_toy_corpus)
print("\nFinal vocabulary from toy corpus:")
print(sorted(toy_vocab_final))


# Simple BPE-based segmentation using learned merges

def apply_bpe(word, merges):
    """
    Segment a word using learned BPE merges.
    Start from characters + _ and repeatedly merge if pair is in merges.
    """
    # Represent word as characters + end-of-word marker
    tokens = list(word) + ["_"]
    # We apply merges greedily in the order they were learned
    for (a, b) in merges:
        i = 0
        new_tokens = []
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == a and tokens[i+1] == b:
                new_tokens.append(a + b)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    return tokens

test_words = ["new", "newer", "lowest", "widest", "newestest"]
print("\n=== BPE segmentation on selected words ===")
for w in test_words:
    seg = apply_bpe(w, toy_merges)
    print(f"{w} -> {seg}")


"""
- Subword tokens solve the OOV problem because even if a full word is unseen,
  its pieces (subwords) are usually in the vocabulary (e.g., new + er_).
- This lets us represent rare or invented words like 'newestest' using known parts.
- Example of meaningful morpheme: 'er_' often aligns with the English comparative/agent suffix.
- BPE learns frequent patterns like 'low' + 'est_' or 'new' + 'er_' that correspond to real morphology.
"""


# 2.3 – Small English paragraph BPE (example)

paragraph = """
Subword tokenization is a powerful technique for handling rare words and complex morphology.
It breaks words into smaller units that can be reused across different contexts.
This helps models generalize better and reduces the out-of-vocabulary problem.
However, subword units do not always align perfectly with linguistic boundaries.
They may split meaningful morphemes or merge unrelated parts.
"""

# Build corpus from paragraph: words with characters + _
para_words = [w for w in paragraph.split() if w.strip()]
para_corpus = [" ".join(list(w) + ["_"]) for w in para_words]

print("\n=== BPE on small English paragraph (about 30 merges) ===")
final_para_corpus, para_merges = learn_bpe(para_corpus, num_merges=30)

# Five most frequent merges (from the learned list)
print("\nFirst 5 merges learned on paragraph:")
for i, m in enumerate(para_merges[:5], 1):
    print(f"  {i}: {m}")

# Longest subword tokens from final paragraph corpus
para_vocab_final = update_vocab_from_corpus(final_para_corpus)
sorted_by_length = sorted(para_vocab_final, key=len, reverse=True)
print("\nFive longest subword tokens from paragraph:")
for tok in sorted_by_length[:5]:
    print("  ", tok)

# Segment 5 different words from the paragraph
segment_words_para = [
    "Subword",        # capitalized
    "tokenization",   # derived form
    "morphology",     # content word
    "vocabulary",     # longer word
    "boundaries"      # plural/inflected form
]

print("\n=== BPE segmentation on 5 words from paragraph ===")
for w in segment_words_para:
    seg = apply_bpe(w, para_merges)
    print(f"{w} -> {seg}")

"""

- The learned subwords include prefixes (sub-, out-), stems (word, vocab), suffixes (-tion, -ary, -ies),
  and sometimes whole words when they are frequent.
- Pros:
  1) Handles rare and derived forms by decomposing them into reusable pieces.
  2) Reduces vocabulary size while still covering many word forms.
- Cons:
  1) Subwords may cut through meaningful morphemes, making linguistic interpretation harder.
  2) Some merges reflect frequency rather than true morphology, merging parts that are not meaningful units.
"""

Toy corpus with end-of-word marker (_):
   l o w _
   l o w _
   l o w _
   l o w _
   l o w _

Initial vocabulary (characters + _):
['_', 'd', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w']

=== Mini BPE on toy corpus (showing first 10 merges) ===
Step 1: top pair = ('e', 'r') (count=9), vocab size = 11
Step 2: top pair = ('er', '_') (count=9), vocab size = 11
Step 3: top pair = ('n', 'e') (count=8), vocab size = 11
Step 4: top pair = ('ne', 'w') (count=8), vocab size = 11
Step 5: top pair = ('l', 'o') (count=7), vocab size = 10
Step 6: top pair = ('lo', 'w') (count=7), vocab size = 10
Step 7: top pair = ('new', 'er_') (count=6), vocab size = 11
Step 8: top pair = ('low', '_') (count=5), vocab size = 12
Step 9: top pair = ('w', 'i') (count=3), vocab size = 11
Step 10: top pair = ('wi', 'd') (count=3), vocab size = 10

Final toy corpus snippet after merges:
   low_
   low_
   low_
   low_
   low_

Learned merges (first 10):
  1: ('e', 'r')
  2: ('er', '_')
  3: ('n', 'e')
  4: ('ne', 'w'

'\n\n- The learned subwords include prefixes (sub-, out-), stems (word, vocab), suffixes (-tion, -ary, -ies),\n  and sometimes whole words when they are frequent.\n- Pros:\n  1) Handles rare and derived forms by decomposing them into reusable pieces.\n  2) Reduces vocabulary size while still covering many word forms.\n- Cons:\n  1) Subwords may cut through meaningful morphemes, making linguistic interpretation harder.\n  2) Some merges reflect frequency rather than true morphology, merging parts that are not meaningful units.\n'

In [ ]:
#Q3.Bayes Rule Applied to Text (based on slide: Bayes’ Rule for documents)

## **1. What each term means: P(c), P(d∣c), P(c∣d)**

### **P(c) — Prior probability of the class**
This is how likely a class is *before* seeing the document.
Examples:
- If 70% of your training documents are “sports,” then **P(sports) = 0.7**.
- It reflects the general frequency of the class in your dataset.


---

### **P(d∣c) — Likelihood of the document given the class**
This is how likely the document’s words are *if* the class were true.
In text classification, this comes from your language model for that class.

Example:
- If the class is “sports,” and the document contains words like *team, score, match*,
  then **P(d∣sports)** will be high.


---

### **P(c∣d) — Posterior probability of the class given the document**
This is what you actually want:
**The probability that the document belongs to class c after reading it.**

Bayes rule computes this using:
- How common the class is (P(c))
- How well the class explains the document (P(d∣c))

This is the final classification score.

---

## **2. Why can the denominator P(d) be ignored?**

Because **P(d)** is the same for every class.

When comparing classes (e.g., sports vs politics), you compute:

\[
P(c \mid d) = \frac{P(c)\,P(d\mid c)}{P(d)}
\]

But **P(d)** does not depend on the class — it’s just “how likely this document is overall.”

So when you compare:

\[
P(c_1 \mid d) \quad \text{vs} \quad P(c_2 \mid d)
\]

the denominator **P(d)** is identical for both.
It cancels out, meaning you only need to compare:

\[
P(c)\,P(d\mid c)
\]

This makes classification faster and simpler.

In [ ]:
#Q4.

## **1. Compute the denominator for add‑1 smoothing**

For the **negative class**, you are given:

- Total token count = **14**
- Vocabulary size = **20**

Add‑1 smoothing uses:

\[
\text{denominator} = \text{total tokens} + |V|
\]

So:

\[
14 + 20 = 34
\]

**Denominator = 34**


## **2. Compute smoothed likelihoods**

### **Case A: “predictable” occurs 2 times in negative documents**

Add‑1 smoothing formula:

\[
P(w \mid -) = \frac{C(w,-) + 1}{14 + 20}
\]

Plug in the values:

\[
P(\text{predictable} \mid -) = \frac{2 + 1}{34} = \frac{3}{34}
\]



### **Case B: “fun” occurs 0 times in negative documents**

\[
P(\text{fun} \mid -) = \frac{0 + 1}{34} = \frac{1}{34}
\]


In [15]:
#Q5.
#1. Tokenize a paragraph

"""Paragraph:
The weather was surprisingly warm today. People were enjoying the sunshine, chatting happily in the park. It felt like the perfect day to relax and unwind."""

naive_tokens = [
"The", "weather", "was", "surprisingly", "warm", "today.",
"People", "were", "enjoying", "the", "sunshine,", "chatting",
"happily", "in", "the", "park.",
"It", "felt", "like", "the", "perfect", "day", "to", "relax", "and", "unwind."
]


"""1B. Manually corrected tokens
Corrections applied
Removed punctuation from tokens

Kept meaningful words intact

Treated sentence‑final punctuation as separate tokens

Ensured clean word boundaries"""

manual_tokens = [
"The", "weather", "was", "surprisingly", "warm", "today", ".",
"People", "were", "enjoying", "the", "sunshine", ",", "chatting",
"happily", "in", "the", "park", ".",
"It", "felt", "like", "the", "perfect", "day", "to", "relax", "and", "unwind", "."
]

#2. Compare with an NLP Tool (spaCy)
["The", "weather", "was", "surprisingly", "warm", "today", ".",
 "People", "were", "enjoying", "the", "sunshine", ",", "chatting",
 "happily", "in", "the", "park", ".",
 "It", "felt", "like", "the", "perfect", "day", "to", "relax", "and", "unwind", "."]

#3. Multiword Expressions (MWEs)
"""Three MWEs in English
New York City

A single named entity; splitting loses meaning.

hot dog

Not “hot” + “dog”; meaning changes completely.

by the way

Fixed idiomatic expression; functions as one discourse marker.

Why treat MWEs as single tokens?
Their meaning is not compositional (cannot be derived from individual words).

They behave like single semantic units.

Helps downstream tasks: NER, translation, sentiment, topic classification."""

'Three MWEs in English\nNew York City\n\nA single named entity; splitting loses meaning.\n\nhot dog\n\nNot “hot” + “dog”; meaning changes completely.\n\nby the way\n\nFixed idiomatic expression; functions as one discourse marker.\n\nWhy treat MWEs as single tokens?\nTheir meaning is not compositional (cannot be derived from individual words).\n\nThey behave like single semantic units.\n\nHelps downstream tasks: NER, translation, sentiment, topic classification.'